# 04 — Case Solution Reuse + Revisi & Retain
**Tujuan:** Prediksi outcome kasus baru dari top-k kasus termirip, lalu tambahkan kasus yang terbukti benar ke dalam case base (Retain).

Algoritma:
- **Majority Vote** — label yang paling banyak di top-k
- **Weighted Similarity** — bobot = skor cosine similarity
- **Revisi & Retain** — kasus baru yang valid dimasukkan ke case base untuk iterasi berikutnya

```
projek-cbr-hukum/
└── data/
    ├── eval/     ← input  (queries.json)
    ├── raw/      ← Retain: tambah case_XXX.txt baru
    ├── processed/← Retain: update cases.csv & cases.json
    └── results/  ← output (predictions.csv, retain_log.csv)
```

In [2]:
import re, json, pickle, warnings
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')

BASE          = Path('..').resolve()
PROCESSED_DIR = BASE / 'data' / 'processed'
EVAL_DIR      = BASE / 'data' / 'eval'
RESULTS_DIR   = BASE / 'data' / 'results'
MODELS_DIR    = BASE / 'models'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Load data & model ───────────────────────────────────────
df = pd.read_csv(PROCESSED_DIR / 'cases.csv')
df['text_combined'] = (
    df['ringkasan_fakta'].fillna('') + ' ' +
    df['pasal'].fillna('')            + ' ' +
    df['amar_putusan'].fillna('')
)

tfidf = pickle.load(open(MODELS_DIR / 'tfidf_vectorizer.pkl', 'rb'))
le    = pickle.load(open(MODELS_DIR / 'label_encoder.pkl',    'rb'))

# Filter ke label yang valid
df_f = df[df['label_vonis'].isin(le.classes_)].copy().reset_index(drop=True)
X_all_tfidf = tfidf.transform(df_f['text_combined'].tolist())
case_ids    = df_f['case_id'].tolist()

# BERT (opsional)
bert_model, bert_embeddings = None, None
emb_path = MODELS_DIR / 'bert_embeddings.npy'
if emb_path.exists():
    try:
        from sentence_transformers import SentenceTransformer
        bert_model      = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
        bert_embeddings = np.load(str(emb_path))
        print(f'BERT dimuat: {bert_embeddings.shape}')
    except Exception as e:
        print(f'BERT tidak tersedia: {e}')

# Dictionary solusi
case_solutions: Dict[str, dict] = {
    row['case_id']: {
        'amar_putusan': row['amar_putusan'],
        'vonis'       : row['vonis'],
        'label_vonis' : row['label_vonis'],
        'no_perkara'  : row['no_perkara'],
    }
    for _, row in df_f.iterrows()
}

print(f'Data: {len(df_f)} kasus | BERT aktif: {bert_model is not None}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8811.84it/s]


BERT dimuat: (60, 384)
Data: 60 kasus | BERT aktif: True


In [3]:
# ════════════════════════════════════════════════════════════
# FUNGSI INTI
# ════════════════════════════════════════════════════════════
STOPWORDS = {
    'yang','dan','di','ke','dari','dengan','untuk','pada','dalam',
    'oleh','ini','itu','telah','tidak','ada','adalah','atau','juga',
    'bahwa','tersebut','nomor','tahun','sebagaimana',
}

def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(t for t in text.split() if t not in STOPWORDS and len(t) > 2)

def retrieve(query: str, k: int = 5, method: str = 'bert') -> List[Tuple[str, float]]:
    if method == 'tfidf' or bert_model is None:
        q_vec = tfidf.transform([preprocess(query)])
        sims  = cosine_similarity(q_vec, X_all_tfidf).flatten()
    else:
        q_emb = bert_model.encode([query[:512]], normalize_embeddings=True, show_progress_bar=False)
        sims  = (q_emb @ bert_embeddings.T).flatten()
    top_idx = np.argsort(sims)[::-1][:k]
    return [(case_ids[i], round(float(sims[i]), 4)) for i in top_idx]

def predict_outcome(query: str, k: int = 5,
                    method: str = 'bert', voting: str = 'weighted') -> dict:
    """
    Prediksi outcome kasus baru.
    method : 'tfidf' | 'bert'
    voting : 'majority' | 'weighted'
    """
    top_k    = retrieve(query, k=k, method=method)
    ids      = [t[0] for t in top_k]
    sims     = [t[1] for t in top_k]
    solutions = [case_solutions.get(cid, {}) for cid in ids]
    labels   = [s.get('label_vonis', 'tidak_diketahui') for s in solutions]

    if voting == 'majority':
        cnt        = Counter(labels)
        pred_label = cnt.most_common(1)[0][0]
        confidence = cnt.most_common(1)[0][1] / len(labels)
    else:  # weighted
        weights = {}
        for lbl, sim in zip(labels, sims):
            weights[lbl] = weights.get(lbl, 0.0) + sim
        pred_label = max(weights, key=weights.get)
        confidence = weights[pred_label] / sum(weights.values()) if weights else 0.0

    best_idx   = next((i for i, l in enumerate(labels) if l == pred_label), 0)
    pred_vonis = solutions[best_idx].get('vonis', 'N/A') if solutions else 'N/A'
    pred_amar  = solutions[best_idx].get('amar_putusan', 'N/A') if solutions else 'N/A'

    return {
        'predicted_label'   : pred_label,
        'predicted_vonis'   : pred_vonis,
        'predicted_amar'    : pred_amar,
        'confidence'        : round(confidence, 3),
        'top_k_case_ids'    : ids,
        'similarity_scores' : sims,
        'label_distribution': dict(Counter(labels)),
    }

print('Fungsi predict_outcome siap')

Fungsi predict_outcome siap


In [4]:
# ════════════════════════════════════════════════════════════
# DEMO MANUAL — 5 KASUS BARU
# ════════════════════════════════════════════════════════════
with open(EVAL_DIR / 'queries.json', encoding='utf-8') as f:
    eval_queries = json.load(f)

default_method = 'bert' if bert_model is not None else 'tfidf'

print('=' * 65)
print('DEMO — PREDIKSI 5 KASUS BARU')
print('=' * 65)
for q in eval_queries[:5]:
    res    = predict_outcome(q['query'], k=5, method=default_method, voting='weighted')
    status = '✓ BENAR' if res['predicted_label'] == q['expected_label'] else '✗ SALAH'
    print(f"\n[{q['query_id']}] {q['description']}")
    print(f"  Aktual    : {q['expected_label']}")
    print(f"  Prediksi  : {res['predicted_label']}  (conf={res['confidence']:.0%})  {status}")
    print(f"  Top-5 IDs : {res['top_k_case_ids']}")
    print(f"  Distribusi: {res['label_distribution']}")

DEMO — PREDIKSI 5 KASUS BARU

[Q001] Penganiayaan ringan
  Aktual    : ringan
  Prediksi  : tidak_diketahui  (conf=80%)  ✗ SALAH
  Top-5 IDs : ['case_037', 'case_004', 'case_047', 'case_055', 'case_011']
  Distribusi: {'tidak_diketahui': 4, 'bebas': 1}

[Q002] Penganiayaan dengan senjata
  Aktual    : sedang
  Prediksi  : tidak_diketahui  (conf=79%)  ✗ SALAH
  Top-5 IDs : ['case_002', 'case_053', 'case_011', 'case_013', 'case_004']
  Distribusi: {'bebas': 1, 'tidak_diketahui': 4}

[Q003] Penganiayaan berujung kematian
  Aktual    : berat
  Prediksi  : tidak_diketahui  (conf=80%)  ✗ SALAH
  Top-5 IDs : ['case_004', 'case_011', 'case_055', 'case_038', 'case_044']
  Distribusi: {'tidak_diketahui': 4, 'bebas': 1}

[Q004] KDRT terhadap anak
  Aktual    : sedang
  Prediksi  : tidak_diketahui  (conf=100%)  ✗ SALAH
  Top-5 IDs : ['case_011', 'case_004', 'case_044', 'case_036', 'case_046']
  Distribusi: {'tidak_diketahui': 5}

[Q005] Perkelahian di tempat umum
  Aktual    : ringan
  Prediksi  :

In [5]:
# ════════════════════════════════════════════════════════════
# PREDIKSI SEMUA QUERY  →  data/results/predictions.csv
# ════════════════════════════════════════════════════════════
records = []
for q in eval_queries:
    r_tf = predict_outcome(q['query'], method='tfidf',           voting='weighted')
    r_bt = predict_outcome(q['query'], method=default_method,    voting='weighted')

    records.append({
        'query_id'           : q['query_id'],
        'description'        : q.get('description', ''),
        'expected_label'     : q['expected_label'],
        'predicted_tfidf'    : r_tf['predicted_label'],
        'predicted_bert'     : r_bt['predicted_label'],
        'predicted_solution' : str(r_bt['predicted_amar'])[:200],
        'confidence_bert'    : r_bt['confidence'],
        'top_5_case_ids'     : str(r_bt['top_k_case_ids']),
        'top_5_similarities' : str(r_bt['similarity_scores']),
        'correct_tfidf'      : r_tf['predicted_label'] == q['expected_label'],
        'correct_bert'       : r_bt['predicted_label'] == q['expected_label'],
    })

df_pred = pd.DataFrame(records)
df_pred.to_csv(RESULTS_DIR / 'predictions.csv', index=False, encoding='utf-8')

acc_tf = df_pred['correct_tfidf'].mean()
acc_bt = df_pred['correct_bert'].mean()
print(f' predictions.csv tersimpan → {RESULTS_DIR}')
print(f'   Demo Accuracy — TF-IDF: {acc_tf:.1%} | BERT: {acc_bt:.1%}')
df_pred[['query_id','expected_label','predicted_tfidf','predicted_bert',
          'confidence_bert','correct_bert']].head(10)

 predictions.csv tersimpan → D:\Kuliah\Tugas Penalaran Komputer\Penalaran-Komputer_SubCpmk3\data\results
   Demo Accuracy — TF-IDF: 60.0% | BERT: 46.7%


,query_id,expected_label,predicted_tfidf,predicted_bert,confidence_bert,correct_bert
0,Q001,ringan,bebas,tidak_diketahui,0.802,False
1,Q002,sedang,tidak_diketahui,tidak_diketahui,0.789,False
2,Q003,berat,tidak_diketahui,tidak_diketahui,0.802,False
3,Q004,sedang,tidak_diketahui,tidak_diketahui,1.000,False
4,Q005,ringan,tidak_diketahui,tidak_diketahui,0.806,False
5,REAL_case_027,bebas,bebas,tidak_diketahui,0.577,False
6,REAL_case_004,tidak_diketahui,tidak_diketahui,tidak_diketahui,1.000,True
7,REAL_case_003,tidak_diketahui,tidak_diketahui,tidak_diketahui,1.000,True
8,REAL_case_041,tidak_diketahui,tidak_diketahui,tidak_diketahui,1.000,True
9,REAL_case_030,tidak_diketahui,tidak_diketahui,tidak_diketahui,1.000,True


In [6]:
# ════════════════════════════════════════════════════════════
# TAHAP REVISI — Validasi sebelum kasus ditambahkan ke case base
# ════════════════════════════════════════════════════════════
# Kriteria REVISI (kasus TIDAK diretain jika):
#   1. Confidence prediksi < threshold (prediksi tidak yakin)
#   2. Max similarity dari top-k terlalu rendah (kasus terlalu berbeda)
#   3. Solusi yang diberikan belum dikonfirmasi benar

RETAIN_CONFIDENCE_THRESHOLD  = 0.60   # min confidence untuk retain
RETAIN_SIMILARITY_THRESHOLD  = 0.25   # min max-similarity top-k
RAW_DIR       = BASE / 'data' / 'raw'
RETAIN_LOG    = RESULTS_DIR / 'retain_log.csv'

def revise(prediction: dict, confirmed_label: str) -> dict:
    """
    Tahap Revisi — cek apakah prediksi layak diretain.

    Parameters
    ----------
    prediction      : output dari predict_outcome()
    confirmed_label : label vonis yang TERBUKTI BENAR (dari user/pakar)

    Returns
    -------
    dict: {should_retain, reason, confirmed_label}
    """
    confidence  = prediction.get('confidence', 0)
    max_sim     = max(prediction.get('similarity_scores', [0]))
    pred_label  = prediction.get('predicted_label', '')

    # Cek 1: confidence cukup?
    if confidence < RETAIN_CONFIDENCE_THRESHOLD:
        return {
            'should_retain' : False,
            'reason'        : f'Confidence {confidence:.0%} < threshold {RETAIN_CONFIDENCE_THRESHOLD:.0%}',
            'confirmed_label': confirmed_label,
        }

    # Cek 2: similarity cukup?
    if max_sim < RETAIN_SIMILARITY_THRESHOLD:
        return {
            'should_retain' : False,
            'reason'        : f'Max similarity {max_sim:.3f} < threshold {RETAIN_SIMILARITY_THRESHOLD}',
            'confirmed_label': confirmed_label,
        }

    # Cek 3: prediksi cocok dengan label yang dikonfirmasi?
    if pred_label != confirmed_label:
        return {
            'should_retain' : False,
            'reason'        : f'Prediksi ({pred_label}) ≠ label konfirmasi ({confirmed_label}) — perlu revisi manual',
            'confirmed_label': confirmed_label,
        }

    return {
        'should_retain' : True,
        'reason'        : 'Lolos semua kriteria validasi',
        'confirmed_label': confirmed_label,
    }

print('Fungsi revise() siap')
print(f'Threshold confidence : {RETAIN_CONFIDENCE_THRESHOLD:.0%}')
print(f'Threshold similarity : {RETAIN_SIMILARITY_THRESHOLD}')


Fungsi revise() siap
Threshold confidence : 60%
Threshold similarity : 0.25


In [7]:
# ════════════════════════════════════════════════════════════
# TAHAP RETAIN — Tambahkan kasus baru ke case base
# ════════════════════════════════════════════════════════════
import re as _re
from datetime import datetime

def retain(
    query_text      : str,
    confirmed_label : str,
    amar_putusan    : str,
    vonis           : str,
    no_perkara      : str = 'N/A',
    tanggal         : str = None,
    pasal           : str = 'N/A',
    dry_run         : bool = False,
) -> dict:
    """
    Retain kasus baru ke dalam case base.

    Alur:
    1. Generate case_id baru (lanjutan dari kasus terakhir)
    2. Simpan teks ke data/raw/case_XXX.txt
    3. Tambahkan baris baru ke data/processed/cases.csv & cases.json
    4. Update X_all_tfidf (in-memory) dan bert_embeddings (jika aktif)
    5. Catat ke data/results/retain_log.csv

    dry_run=True: simulasi tanpa menulis file
    """
    global df_f, X_all_tfidf, bert_embeddings, case_ids, case_solutions

    # ── Generate case_id baru ─────────────────────────────────
    existing = sorted(RAW_DIR.glob('case_*.txt'))
    next_num  = len(existing) + 1
    new_id    = f'case_{next_num:03d}'

    # Pastikan tidak bentrok
    while new_id in case_ids:
        next_num += 1
        new_id = f'case_{next_num:03d}'

    tanggal = tanggal or datetime.now().strftime('%Y-%m-%d')

    new_row = {
        'case_id'        : new_id,
        'no_perkara'     : no_perkara,
        'tanggal'        : tanggal,
        'jenis_perkara'  : 'Pidana Umum - Penganiayaan',
        'terdakwa'       : 'N/A',
        'penuntut_umum'  : 'N/A',
        'pasal'          : pasal,
        'vonis'          : vonis,
        'amar_putusan'   : amar_putusan,
        'ringkasan_fakta': query_text[:600],
        'barang_bukti'   : 'N/A',
        'word_count'     : len(query_text.split()),
        'label_vonis'    : confirmed_label,
        'text_combined'  : query_text,
        'source'         : 'retained',       # penanda kasus hasil retain
    }

    print(f'[RETAIN] case_id baru: {new_id} | label: {confirmed_label} | dry_run={dry_run}')

    if dry_run:
        print('  → dry_run aktif: tidak ada file yang ditulis.')
        return {'status': 'dry_run', 'case_id': new_id, 'row': new_row}

    # ── 1. Simpan teks ke data/raw/ ───────────────────────────
    raw_path = RAW_DIR / f'{new_id}.txt'
    raw_path.write_text(query_text, encoding='utf-8')

    # ── 2. Update cases.csv ───────────────────────────────────
    csv_path  = PROCESSED_DIR / 'cases.csv'
    df_csv    = pd.read_csv(csv_path)
    # Tambah kolom 'source' jika belum ada
    if 'source' not in df_csv.columns:
        df_csv['source'] = 'original'
    df_csv = pd.concat([df_csv, pd.DataFrame([new_row])], ignore_index=True)
    df_csv.to_csv(csv_path, index=False, encoding='utf-8')

    # ── 3. Update cases.json ──────────────────────────────────
    json_path = PROCESSED_DIR / 'cases.json'
    with open(json_path, encoding='utf-8') as f:
        cases_list = json.load(f)
    cases_list.append(new_row)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(cases_list, f, ensure_ascii=False, indent=2)

    # ── 4. Update in-memory state ─────────────────────────────
    new_df_row = pd.DataFrame([new_row])
    df_f       = pd.concat([df_f, new_df_row], ignore_index=True)
    case_ids.append(new_id)

    # Update TF-IDF matrix
    new_vec     = tfidf.transform([query_text])
    import scipy.sparse as sp
    X_all_tfidf = sp.vstack([X_all_tfidf, new_vec])

    # Update BERT embeddings
    if bert_model is not None and bert_embeddings is not None:
        new_emb       = bert_model.encode([query_text[:512]], normalize_embeddings=True,
                                          show_progress_bar=False)
        bert_embeddings = np.vstack([bert_embeddings, new_emb])
        np.save(str(MODELS_DIR / 'bert_embeddings.npy'), bert_embeddings)

    # Update case_solutions
    case_solutions[new_id] = {
        'amar_putusan': amar_putusan,
        'vonis'       : vonis,
        'label_vonis' : confirmed_label,
        'no_perkara'  : no_perkara,
    }

    # ── 5. Catat ke retain_log.csv ────────────────────────────
    log_entry = pd.DataFrame([{
        'timestamp'      : datetime.now().isoformat(),
        'case_id'        : new_id,
        'confirmed_label': confirmed_label,
        'no_perkara'     : no_perkara,
        'vonis'          : vonis,
        'text_preview'   : query_text[:100],
    }])
    if RETAIN_LOG.exists():
        log_entry.to_csv(RETAIN_LOG, mode='a', header=False, index=False)
    else:
        log_entry.to_csv(RETAIN_LOG, index=False)

    print(f'  Tersimpan: {raw_path.name}')
    print(f'  cases.csv & cases.json diperbarui')
    print(f'  In-memory state diperbarui (total kasus: {len(case_ids)})')
    print(f'  Retain log: {RETAIN_LOG.name}')

    return {'status': 'success', 'case_id': new_id, 'row': new_row}

print('Fungsi retain() siap')


Fungsi retain() siap


In [8]:
# ════════════════════════════════════════════════════════════
# DEMO REVISI & RETAIN — 3 Skenario (Domain: Penganiayaan)
# ════════════════════════════════════════════════════════════
# Skenario 1: Kasus yang LOLOS retain (confidence tinggi, prediksi benar)
# Skenario 2: Kasus yang DITOLAK (confidence terlalu rendah)
# Skenario 3: Kasus yang DITOLAK (prediksi ≠ label konfirmasi)

skenario = [
    {
        'label'           : 'Skenario 1 — Lolos Retain',
        'query'           : 'terdakwa memukul korban dengan benda tumpul berulang kali '
                            'mengakibatkan luka berat di bagian kepala, '
                            'melanggar pasal 351 ayat 2 kuhp',
        'confirmed_label' : 'sedang',
        'amar'            : 'Menjatuhkan pidana penjara selama 4 tahun kepada terdakwa',
        'vonis'           : 'pidana penjara selama 4 tahun',
        'no_perkara'      : '999/Pid/2024/PN.Mlg',
        'dry_run'         : True,   # ← ubah ke False untuk benar-benar menyimpan
    },
    {
        'label'           : 'Skenario 2 — Ditolak (confidence rendah)',
        'query'           : 'seseorang ditemukan di dekat lokasi kejadian',
        'confirmed_label' : 'ringan',
        'amar'            : 'N/A',
        'vonis'           : 'N/A',
        'no_perkara'      : 'N/A',
        'dry_run'         : True,
    },
    {
        'label'           : 'Skenario 3 — Ditolak (prediksi ≠ konfirmasi)',
        'query'           : 'terdakwa menganiaya korban secara sadis hingga meninggal dunia '
                            'menggunakan senjata tajam pasal 351 ayat 3 kuhp',
        'confirmed_label' : 'ringan',   # sengaja salah → pasti ditolak
        'amar'            : 'N/A',
        'vonis'           : 'N/A',
        'no_perkara'      : 'N/A',
        'dry_run'         : True,
    },
]

print('=' * 65)
print('DEMO REVISI & RETAIN — Pidana Umum Penganiayaan')
print('=' * 65)

default_method = 'bert' if bert_model is not None else 'tfidf'

for s in skenario:
    print(f"\n▶ {s['label']}")
    print(f"  Query: {s['query'][:70]}...")

    # Prediksi dulu
    pred = predict_outcome(s['query'], k=5, method=default_method, voting='weighted')
    print(f"  Prediksi: {pred['predicted_label']} (conf={pred['confidence']:.0%}) "
          f"| max_sim={max(pred['similarity_scores']):.3f}")

    # Revisi — validasi apakah layak retain
    rev = revise(pred, confirmed_label=s['confirmed_label'])
    print(f"  Revisi  : should_retain={rev['should_retain']} | {rev['reason']}")

    # Retain (hanya jika lolos revisi)
    if rev['should_retain']:
        result = retain(
            query_text      = s['query'],
            confirmed_label = s['confirmed_label'],
            amar_putusan    = s['amar'],
            vonis           = s['vonis'],
            no_perkara      = s['no_perkara'],
            dry_run         = s['dry_run'],
        )
        print(f"  Retain  : {result['status']} → {result['case_id']}")
    else:
        print(f"  Retain  : DILEWATI — kasus tidak dimasukkan ke case base")

print('\n' + '=' * 65)
print(f'Total kasus aktif (in-memory): {len(case_ids)}')


DEMO REVISI & RETAIN — Pidana Umum Penganiayaan

▶ Skenario 1 — Lolos Retain
  Query: terdakwa memukul korban dengan benda tumpul berulang kali mengakibatka...
  Prediksi: tidak_diketahui (conf=80%) | max_sim=0.564
  Revisi  : should_retain=False | Prediksi (tidak_diketahui) ≠ label konfirmasi (sedang) — perlu revisi manual
  Retain  : DILEWATI — kasus tidak dimasukkan ke case base

▶ Skenario 2 — Ditolak (confidence rendah)
  Query: seseorang ditemukan di dekat lokasi kejadian...
  Prediksi: tidak_diketahui (conf=85%) | max_sim=0.436
  Revisi  : should_retain=False | Prediksi (tidak_diketahui) ≠ label konfirmasi (ringan) — perlu revisi manual
  Retain  : DILEWATI — kasus tidak dimasukkan ke case base

▶ Skenario 3 — Ditolak (prediksi ≠ konfirmasi)
  Query: terdakwa menganiaya korban secara sadis hingga meninggal dunia menggun...
  Prediksi: bebas (conf=60%) | max_sim=0.588
  Revisi  : should_retain=False | Prediksi (bebas) ≠ label konfirmasi (ringan) — perlu revisi manual
  Retain  : 

In [9]:
# ════════════════════════════════════════════════════════════
# BATCH RETAIN — dari predictions.csv yang sudah dikonfirmasi benar
# ════════════════════════════════════════════════════════════
# Ambil semua prediksi yang BENAR (correct_bert=True) dari test set
# sebagai kandidat untuk diretain ke case base

df_pred = pd.read_csv(RESULTS_DIR / 'predictions.csv')

# Kandidat retain: prediksi benar + hanya query dari data nyata (REAL_)
# Ambil prediksi benar dari kasus nyata penganiayaan
candidates = df_pred[
    (df_pred['correct_bert'] == True) &
    (df_pred['query_id'].str.startswith('REAL_'))
].copy()

print(f'Kandidat retain: {len(candidates)} kasus (prediksi benar dari test set)')
print()

# Muat query teks asli dari queries.json
with open(EVAL_DIR / 'queries.json', encoding='utf-8') as f:
    all_queries = json.load(f)
query_map = {q['query_id']: q for q in all_queries}

retain_results = []
DRY_RUN_BATCH  = True   # ← ubah ke False untuk benar-benar menyimpan semua

for _, row in candidates.head(5).iterrows():   # demo maks 5
    q_info = query_map.get(row['query_id'], {})
    if not q_info:
        continue

    pred = predict_outcome(q_info['query'], k=5,
                           method=default_method, voting='weighted')
    rev  = revise(pred, confirmed_label=row['expected_label'])

    status = 'SKIP'
    if rev['should_retain']:
        result = retain(
            query_text      = q_info['query'],
            confirmed_label = row['expected_label'],
            amar_putusan    = row.get('predicted_solution', 'N/A'),
            vonis           = 'dari retain batch',
            no_perkara      = f"RETAIN_{row['query_id']}",
            dry_run         = DRY_RUN_BATCH,
        )
        status = result['status']

    retain_results.append({
        'query_id'       : row['query_id'],
        'confirmed_label': row['expected_label'],
        'should_retain'  : rev['should_retain'],
        'reason'         : rev['reason'],
        'status'         : status,
    })

df_retain = pd.DataFrame(retain_results)
print('\n--- Ringkasan Batch Retain ---')
print(df_retain[['query_id','confirmed_label','should_retain','status']].to_string(index=False))
print(f'\nTotal retained: {df_retain["should_retain"].sum()}/{len(df_retain)}')
print(f'Total kasus aktif sekarang: {len(case_ids)}')


Kandidat retain: 7 kasus (prediksi benar dari test set)

[RETAIN] case_id baru: case_061 | label: tidak_diketahui | dry_run=True
  → dry_run aktif: tidak ada file yang ditulis.
[RETAIN] case_id baru: case_061 | label: tidak_diketahui | dry_run=True
  → dry_run aktif: tidak ada file yang ditulis.
[RETAIN] case_id baru: case_061 | label: tidak_diketahui | dry_run=True
  → dry_run aktif: tidak ada file yang ditulis.
[RETAIN] case_id baru: case_061 | label: tidak_diketahui | dry_run=True
  → dry_run aktif: tidak ada file yang ditulis.
[RETAIN] case_id baru: case_061 | label: tidak_diketahui | dry_run=True
  → dry_run aktif: tidak ada file yang ditulis.

--- Ringkasan Batch Retain ---
     query_id confirmed_label  should_retain  status
REAL_case_004 tidak_diketahui           True dry_run
REAL_case_003 tidak_diketahui           True dry_run
REAL_case_041 tidak_diketahui           True dry_run
REAL_case_030 tidak_diketahui           True dry_run
REAL_case_051 tidak_diketahui           True d